# PHASE 4 — XGBoost (Modèle principal)
## FRAUDX — Détection de fraude bancaire par IA au Togo

**Objectif :** Implémenter, optimiser et évaluer XGBoost comme modèle principal de classification (Niveau 2).

**Jours 15-24 du plan**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (confusion_matrix, classification_report,
                             f1_score, recall_score, precision_score,
                             roc_auc_score, average_precision_score,
                             precision_recall_curve)
import xgboost as xgb
from imblearn.over_sampling import SMOTE

sns.set_style('whitegrid')

In [ ]:
# === CHARGEMENT ===
from google.colab import drive
drive.mount('/content/drive')

data_path = '/content/drive/MyDrive/FRAUDX/data/'

X_train = pd.read_pickle(f'{data_path}X_train.pkl')
X_test = pd.read_pickle(f'{data_path}X_test.pkl')
y_train = pd.read_pickle(f'{data_path}y_train.pkl')
y_test = pd.read_pickle(f'{data_path}y_test.pkl')

if isinstance(y_train, pd.DataFrame):
    y_train = y_train.values.ravel()
if isinstance(y_test, pd.DataFrame):
    y_test = y_test.values.ravel()

print(f'X_train : {X_train.shape}, y_train : {y_train.shape}')
print(f'Fraude Train : {y_train.mean()*100:.2f}%')
print(f'Fraude Test : {y_test.mean()*100:.2f}%')

---
## JOUR 15 — Premier modèle + Stratégies anti-déséquilibre

**Objectif :** Tester SMOTE, scale_pos_weight, et leur combinaison — comparer les résultats.

In [ ]:
def evaluate_model(model, X_test, y_test, label='Modèle'):
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    
    f1 = f1_score(y_test, preds)
    recall = recall_score(y_test, preds)
    precision = precision_score(y_test, preds)
    auc_pr = average_precision_score(y_test, probs)
    
    print(f'=== {label} ===')
    print(f'  F1-Score    : {f1:.4f}')
    print(f'  Recall      : {recall:.4f}')
    print(f'  Precision   : {precision:.4f}')
    print(f'  AUC-PR      : {auc_pr:.4f}')
    print()
    return {'f1': f1, 'recall': recall, 'precision': precision, 'auc_pr': auc_pr}

In [ ]:
# === Configuration 1 : SMOTE seul ===
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

model_smote = xgb.XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    random_state=42, use_label_encoder=False, eval_metric='logloss'
)
model_smote.fit(X_train_smote, y_train_smote)
metrics_smote = evaluate_model(model_smote, X_test, y_test, 'SMOTE seul')

In [ ]:
# === Configuration 2 : scale_pos_weight seul ===
ratio = (y_train == 0).sum() / (y_train == 1).sum()

model_weight = xgb.XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    scale_pos_weight=ratio,
    random_state=42, use_label_encoder=False, eval_metric='logloss'
)
model_weight.fit(X_train, y_train)
metrics_weight = evaluate_model(model_weight, X_test, y_test, 'scale_pos_weight seul')

In [ ]:
# === Configuration 3 : SMOTE + scale_pos_weight ===
model_both = xgb.XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    scale_pos_weight=ratio,
    random_state=42, use_label_encoder=False, eval_metric='logloss'
)
model_both.fit(X_train_smote, y_train_smote)
metrics_both = evaluate_model(model_both, X_test, y_test, 'SMOTE + scale_pos_weight')

In [ ]:
# === Tableau comparatif des 3 stratégies ===
comparison = pd.DataFrame({
    'SMOTE seul': metrics_smote,
    'scale_pos_weight seul': metrics_weight,
    'SMOTE + scale_pos_weight': metrics_both
}).T

print('=== TABLEAU COMPARATIF DES STRATÉGIES ANTI-DÉSÉQUILIBRE ===')
print('\n' + comparison.to_string())

# Meilleure configuration
best_config = comparison['f1'].idxmax()
print(f'\n✅ Meilleure configuration (F1) : {best_config}')
print('\nCe tableau sera présenté dans la section 3.3 du mémoire.')

---
## JOURS 16-19 — Optimisation Optuna

**Objectif :** Trouver les meilleurs hyperparamètres pour maximiser le F1-Score.

In [ ]:
# === Définition de l'objectif Optuna ===
def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42,
        'use_label_encoder': False,
        'eval_metric': 'logloss'
    }
    
    # Utiliser SMOTE sur la meilleure config identifiée
    model = xgb.XGBClassifier(**param)
    
    # Validation croisée StratifiedKFold
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_smote, y_train_smote,
                             cv=skf, scoring='f1', n_jobs=-1)
    
    return scores.mean()

In [ ]:
# === Lancement d'Optuna ===
print('=== OPTIMISATION OPTUNA ===')
print('Nombre d\'essais : 30 (augmenter pour meilleur résultat)')
print('Objectif : maximiser F1-Score\n')

study = optuna.create_study(direction='maximize', study_name='fraud_xgb')
study.optimize(objective, n_trials=30, show_progress_bar=True)

In [ ]:
# === Résultats ===
print('=== MEILLEURS HYPERPARAMÈTRES ===')
print(f"Best F1 (CV) : {study.best_value:.4f}")
print(f"Best params  : {study.best_params}")

# Tracer l'historique
fig = optuna.visualization.plot_optimization_history(study)
fig.show()

---
## JOUR 20 — Modèle final et ajustement du seuil

**Objectif :** Entraîner le modèle final avec les meilleurs hyperparamètres et ajuster le seuil de décision.

In [ ]:
# === Modèle final ===
best_params = study.best_params
best_params['random_state'] = 42
best_params['use_label_encoder'] = False
best_params['eval_metric'] = 'logloss'

model_final = xgb.XGBClassifier(**best_params)
model_final.fit(X_train_smote, y_train_smote)

print('✅ Modèle final entraîné avec les meilleurs hyperparamètres')

In [ ]:
# === Ajustement du seuil via courbe Precision-Recall ===
y_probs = model_final.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)

# Trouver le seuil qui maximise F1
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_threshold_idx = f1_scores.argmax()
best_threshold = thresholds[best_threshold_idx] if best_threshold_idx < len(thresholds) else 0.5

print(f'Meilleur seuil (max F1) : {best_threshold:.4f}')
print(f'F1 au meilleur seuil : {f1_scores[best_threshold_idx]:.4f}')

In [ ]:
# === Évaluation finale avec le meilleur seuil ===
y_pred_optimal = (y_probs >= best_threshold).astype(int)

print('=== PERFORMANCES FINALES ===')
print(f'F1-Score        : {f1_score(y_test, y_pred_optimal):.4f}')
print(f'Recall          : {recall_score(y_test, y_pred_optimal):.4f}')
print(f'Precision       : {precision_score(y_test, y_pred_optimal):.4f}')
print(f'AUC-PR          : {average_precision_score(y_test, y_probs):.4f}')
print(f'AUC-ROC         : {roc_auc_score(y_test, y_probs):.4f}')

In [ ]:
# === Matrice de confusion ===
cm = confusion_matrix(y_test, y_pred_optimal)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Non fraude', 'Fraude'],
            yticklabels=['Non fraude', 'Fraude'])
ax.set_xlabel('Prédiction')
ax.set_ylabel('Réel')
ax.set_title('Figure 3.x — Matrice de confusion — XGBoost final', fontweight='bold')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'Vrais négatifs (TN) : {tn}')
print(f'Faux positifs (FP)  : {fp}')
print(f'Faux négatifs (FN)  : {fn}  ← HS1 : minimiser ce chiffre')
print(f'Vrais positifs (TP) : {tp}')

In [ ]:
# === Sauvegarde ===
joblib.dump(model_final, 'models/xgb_model.pkl')
np.save('models/best_threshold.npy', best_threshold)
print('✅ Modèle XGBoost final sauvegardé dans models/xgb_model.pkl')
print(f'✅ Seuil optimal sauvegardé : {best_threshold:.4f}')

---
## JOURS 21-24 — Analyse des erreurs et robustesse

**Objectif :** Caractériser les faux positifs et faux négatifs pour le diagnostic (Chapitre IV).

In [ ]:
# === Caractéristiques des erreurs ===
# Recharger X_test original pour analyse
X_test_orig = pd.read_pickle(f'{data_path}X_featured.pkl').loc[X_test.index]

results = X_test_orig.copy()
results['y_true'] = y_test
results['y_pred'] = y_pred_optimal
results['y_prob'] = y_probs

# Faux positifs
fp_df = results[(results['y_true'] == 0) & (results['y_pred'] == 1)]
# Faux négatifs
fn_df = results[(results['y_true'] == 1) & (results['y_pred'] == 0)]

print(f'=== ANALYSE DES ERREURS ===')
print(f'Faux positifs (FP) : {len(fp_df)}')
print(f'Faux négatifs (FN) : {len(fn_df)}')

In [ ]:
# === Analyse par tranche de montant ===
if 'TransactionAmt' in results.columns:
    results['amt_bin'] = pd.qcut(results['TransactionAmt'], q=5, labels=['Très faible', 'Faible', 'Moyen', 'Élevé', 'Très élevé'])
    
    print('=== Performance par tranche de montant ===')
    for bin_name in ['Très faible', 'Faible', 'Moyen', 'Élevé', 'Très élevé']:
        subset = results[results['amt_bin'] == bin_name]
        if len(subset) > 0:
            f1 = f1_score(subset['y_true'], subset['y_pred'])
            recall = recall_score(subset['y_true'], subset['y_pred'])
            print(f'  {bin_name:12s} | F1={f1:.3f} | Recall={recall:.3f} | N={len(subset)}')

In [ ]:
# === Figure — Distribution des probabilités ===
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(y_probs[y_test == 0], bins=50, alpha=0.6, label='Non fraude', color='steelblue', density=True)
ax.hist(y_probs[y_test == 1], bins=50, alpha=0.6, label='Fraude', color='crimson', density=True)
ax.axvline(best_threshold, color='black', linestyle='--', label=f'Seuil optimal = {best_threshold:.3f}')
ax.set_xlabel('Probabilité prédite (fraude)')
ax.set_ylabel('Densité')
ax.set_title('Figure 3.x — Distribution des probabilités prédites par classe', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## Synthèse Phase 4 — Éléments pour le mémoire

### Tableaux pour le mémoire :
1. **Tableau 3.x** — Comparaison des 3 stratégies anti-déséquilibre (SMOTE, scale_pos_weight, combiné)
2. **Tableau 3.x** — Meilleurs hyperparamètres Optuna
3. **Tableau 3.x** — Performances finales (F1, Recall, Precision, AUC-PR, AUC-ROC)

### Figures pour le mémoire :
1. **Figure 3.x** — Matrice de confusion — XGBoost final
2. **Figure 3.x** — Distribution des probabilités prédites par classe
3. **Figure 3.x** — Courbe Precision-Recall (optionnel)

### Métriques clés pour HS1 :
- F1-Score : variable (attendu > 0.80)
- Recall : variable (attendu > 0.85) → directement lié à HS1 (taux de faux négatifs)
- Faux négatifs : variable (à minimiser)